In [ ]:
# Questão 1 – Tratamento de dados.

#A)

import pandas as pd

# Biblioteca tabulate para visualizar os dados como uma planilha real
from tabulate import tabulate

# Variavel para carregar ambos os arquivos (locais e offshore).
dados1 = pd.read_excel('retornos_dados_locais.xlsx',skiprows=1)
dados2 = pd.read_excel('retornos_dados_offshore.xlsx',skiprows=1)

# Ajustando nomes e tipos
dados1.rename(columns={dados1.columns[0]: 'Data'}, inplace=True)
dados2.rename(columns={dados2.columns[0]: 'Data'}, inplace=True)

# Variavel para Armazenar o merge das tabeleas
dados_final = pd.merge(dados1, dados2, on='Data', how='outer')

#Converter a coluna Data para o formato datetime real
dados_final['Data'] = pd.to_datetime(dados_final['Data'], errors='coerce')

#Converter todas as outras colunas para numérico (float)
colunas_ativos = dados_final.columns.drop('Data')
dados_final[colunas_ativos] = dados_final[colunas_ativos].apply(pd.to_numeric, errors='coerce')

print(tabulate(dados_final.head(), headers='keys', tablefmt='psql'))

In [ ]:
#B)

# Guandando as taxas nas respectivas variaveis
peso_RFI = 10 / 100
peso_ALTL = 2 / 100
peso_PosFixa = 31.5 / 100
peso_Inflacao = 22.5 / 100
peso_RetAb = 16.5 / 100
peso_RvLocal = 5 / 100
peso_RVglob = 7.5 / 100
peso_Flls = 5 / 100

# Criando variavel de ativos presentes nos 3 portifolios 
fixos = ( 
  (peso_RFI * dados_final['RF Global']) 
+ (peso_ALTL * dados_final['Dólar']) 
+ (peso_PosFixa * dados_final['CDI']) 
+ (peso_RetAb * dados_final['IHFA']) 
+ (peso_RvLocal * dados_final['Ibovespa']) 
+ (peso_RVglob * dados_final['RV Global'])
+ (peso_Flls * dados_final['IFIX'])
)
# Criando variaveis para o IMA-B nos 3 portifolios 
dados_final['Portifolio_1'] = (fixos + peso_Inflacao * dados_final['IMA-B'])
dados_final['Portifolio_2'] = (fixos + peso_Inflacao * dados_final['IMA-B 5'])
dados_final['Portifolio_3'] = (fixos + peso_Inflacao * dados_final['IMA-B 5+'])


# Cálculo do retorno acumulado total
acumulado = (1 + dados_final[['Portifolio_1', 'Portifolio_2', 'Portifolio_3']]).prod() - 1

# Criando uma lista para o tabulate
resumo = [
    ["Portfólio 1 (IMA-B)", acumulado['Portifolio_1']],
    ["Portfólio 2 (IMA-B 5)", acumulado['Portifolio_2']],
    ["Portfólio 3 (IMA-B 5+)", acumulado['Portifolio_3']]
]

print(tabulate(resumo, headers=["Estratégia", "Retorno Total"], tablefmt='psql', stralign="left", numalign="right", floatfmt=".2%"))


In [ ]:
#C)

print(len(dados_final))

Rt_anualizado = ( 1 + dados_final['CDI'] * )